# 🤖 Customer Churn Prediction — Model Training & Evaluation

**Project**: Customer Churn Prediction  
**Dataset**: IBM Telco Customer Churn  
**Author**: Data Science Team

---

## Objectives
1. Preprocess features (encoding, scaling, feature engineering)
2. Train 7 ML models
3. Compare all models with metrics table
4. Evaluate best model (confusion matrix, ROC, PR curves)
5. Run SHAP explainability
6. Save trained artifacts

In [ ]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

from data_loader import load_and_clean_data
from preprocessing import preprocess_data
from model_training import train_and_select_best, get_model_definitions
from evaluation import ModelEvaluator
from explainability import ChurnExplainer
from utils import CONFIG

plt.style.use('dark_background')
print('✅ All modules loaded!')

---
## Step 1: Load & Preprocess Data

In [ ]:
# Load cleaned data
df, loader = load_and_clean_data()
print(f'Dataset shape: {df.shape}')
print(f'Churn rate: {df["Churn"].mean()*100:.2f}%')

In [ ]:
# Preprocess
X_train, X_test, y_train, y_test, preprocessor = preprocess_data(df)

print(f'\nPreprocessing complete:')
print(f'  Total features: {len(preprocessor.feature_names)}')
print(f'  X_train: {X_train.shape}')
print(f'  X_test:  {X_test.shape}')
print(f'  Train churn rate: {y_train.mean()*100:.2f}%')
print(f'  Test churn rate:  {y_test.mean()*100:.2f}%')

# Save preprocessor
preprocessor.save()
print('\n✅ Preprocessor saved!')

In [ ]:
# Show feature names
print(f'\nAll {len(preprocessor.feature_names)} features:')
for i, feat in enumerate(preprocessor.feature_names):
    print(f'  {i+1:2d}. {feat}')

---
## Step 2: Train All Models

In [ ]:
# Train all 7 models
print('Training 7 models... (this may take 2-5 minutes)')
best_name, best_model, results_df, trainer = train_and_select_best(
    X_train, X_test, y_train, y_test, tune=False
)

print('\n' + '='*70)
print('MODEL COMPARISON TABLE')
print('='*70)
display(results_df.style.highlight_max(
    subset=['Accuracy', 'F1 Score', 'ROC-AUC', 'CV F1 Mean'],
    color='rgba(0, 212, 255, 0.3)'
))

In [ ]:
# Model comparison chart
ModelEvaluator.plot_model_comparison(results_df, save=True)
print(f'\n🏆 Best Model: {best_name}')

---
## Step 3: Evaluate Best Model

In [ ]:
# Full evaluation
evaluator = ModelEvaluator(best_model, model_name=best_name)
eval_report = evaluator.full_evaluation_report(
    X_train, X_test, y_train, y_test,
    all_models=trainer.trained_models,
    save_plots=True
)

print('\nMETRICS SUMMARY')
for k, v in eval_report['metrics'].items():
    if isinstance(v, (int, float)):
        print(f'  {k:<30}: {v:.2f}%' if isinstance(v, float) else f'  {k:<30}: {v}')

In [ ]:
# Show classification report
print('\nDETAILED CLASSIFICATION REPORT')
print('='*60)
print(eval_report['classification_report'])

In [ ]:
# Cross-validation results
cv = eval_report['cv_results']
print(f'\nCROSS-VALIDATION RESULTS (5-Fold F1)')
print('='*40)
print(f'  Scores: {[f"{s:.3f}" for s in cv["scores"]]}') 
print(f'  Mean:   {cv["mean"]:.2f}%')
print(f'  Std:    {cv["std"]:.2f}%')
print(f'  Range:  {cv["min"]:.2f}% – {cv["max"]:.2f}%')

---
## Step 4: SHAP Explainability

In [ ]:
# Run SHAP explainability
print('Running SHAP explainability... (may take 1-3 minutes)')

explainer = ChurnExplainer(
    model=best_model,
    feature_names=preprocessor.feature_names,
    model_name=best_name
)

explainer.run_explainability_pipeline(X_train, X_test, save=True)
print('✅ SHAP plots saved to reports/figures/')

In [ ]:
# Explain a specific customer
print('\nEXPLAINING PREDICTION FOR CUSTOMER #1')
print('='*60)
customer_explanation = explainer.explain_customer(X_test.iloc[[0]])
display(customer_explanation)

---
## Step 5: Summary & Next Steps

In [ ]:
print('='*70)
print('PIPELINE COMPLETE — SUMMARY')
print('='*70)
print(f'  Best Model:   {best_name}')
m = eval_report['metrics']
print(f'  Accuracy:     {m["Accuracy"]:.2f}%')
print(f'  F1 Score:     {m["F1 Score"]:.2f}%')
print(f'  ROC-AUC:      {m["ROC-AUC"]:.2f}%')
print(f'  Precision:    {m["Precision"]:.2f}%')
print(f'  Recall:       {m["Recall (Sensitivity)"]:.2f}%')
print()
print('  Artifacts saved:')
print(f'    models/churn_model.pkl')
print(f'    models/scaler.pkl')
print(f'    models/feature_names.json')
print(f'    reports/figures/ (all plots)')
print()
print('  Next Steps:')
print('    → streamlit run app/app.py  (launch dashboard)')
print('    → python src/predict.py     (test predictions)')
print('='*70)